# LoRA Fine-tuning Bench for Sanskrit POS Tagging

Same task as `indicbert_finetune.ipynb` (word-level POS classification: NOUN / VERB / PRON / PART / IND, with verse context), but trained with **LoRA** (Low-Rank Adaptation) instead of full fine-tuning, and built to **bench three base models** from a single `MODEL_NAME` switch:

| `MODEL_NAME`                | Architecture | Notes |
|-----------------------------|--------------|-------|
| `ai4bharat/indic-bert`      | ALBERT       | The original baseline. ALBERT *shares* weights across layers, so LoRA's upside is muted here. |
| `google/muril-base-cased`   | BERT         | Strongest Devanagari/Indic coverage; usually the best for Sanskrit. |
| `xlm-roberta-base`          | RoBERTa      | Strong multilingual baseline. |
| `l3cube-pune/hindi-bert-v2`                   | BERT         | Trained on large Hindi+Devanagari corpus; script-aware, lighter than MuRIL. |
| `ai4bharat/IndicBERTv2-MLM-Sam-TLM`           | BERT         | Direct successor to indic-bert; 40× more Indic data, full BERT (not ALBERT) so LoRA is fully effective. |

**Why LoRA:** it freezes the pretrained weights and trains tiny low-rank adapter matrices (typically <1% of all parameters). On a T4 this means much lower memory and faster training, at near full-fine-tune quality.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

## Step 1 — Install dependencies

`torchao` is uninstalled first because Colab ships a version recent `peft` rejects.

In [ ]:
!pip uninstall -y torchao -q
!pip install -q transformers datasets accelerate peft indic-transliteration scikit-learn

## Step 2 — Upload the JSON dataset

1. On your computer, zip the entire `json/` folder → name it `json_files.zip`.
2. Run the cell below, click **Choose Files**, select `json_files.zip`.
3. Wait for upload + extraction to finish.

In [ ]:
from google.colab import files
import zipfile, os

print("Upload your json_files.zip ...")
uploaded = files.upload()

os.makedirs('json_files', exist_ok=True)
for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as zf:
            zf.extractall('json_files/')
        print(f"Extracted {fname}")

import glob
json_paths = glob.glob('json_files/**/*.json', recursive=True) + glob.glob('json_files/*.json')
json_paths = [p for p in json_paths if 'bhagavata_book' in os.path.basename(p)]
print(f"Found {len(json_paths)} chapter JSON files")

## Step 3 — Load and parse data (disambiguating loader)

Identical to the baseline notebook: cleans verses, converts IAST → Devanagari, maps lexical categories to POS labels, and applies high-precision closed-class / suffix / finite-verb overrides before falling back to the analyzer's first candidate.

In [ ]:
import json, re, random
from collections import Counter
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

_VERSE_NUM = re.compile(r'॥[\s\d०-९]+॥')

def clean_verse(text: str) -> str:
    text = _VERSE_NUM.sub('', text)
    return re.sub(r'\s+', ' ', text).strip()

def iast_to_devanagari(text: str) -> str:
    try:
        return transliterate(text, sanscript.IAST, sanscript.DEVANAGARI)
    except Exception:
        return text


def simplify_pos(lex_catg: str):
    lex = lex_catg.lower().strip()
    if 'pronoun' in lex:                                          return 'PRON'
    if 'noun' in lex:                                             return 'NOUN'
    if 'verb' in lex:                                             return 'VERB'
    if 'particle' in lex or 'krdanta' in lex or 'kṛdanta' in lex: return 'PART'
    if 'indecl' in lex or 'avyaya' in lex or 'prefix' in lex:     return 'IND'
    return None


PRON_WORDS = {
    'tad','etad','idam','adas','asmad','yuṣmad','kim','yad','eṣa','ena',
    'ayam','asau','ima','sa','sā','tat','etat','aham','tvam','mad','ka',
}
IND_WORDS = {
    'ca','vā','hi','eva','iva','api','tu','sma','uta','atha','atho',
    'vai','kila','khalu','nu','nūnam','iti','na','mā','yathā','tathā','evam',
    'yatra','tatra','kutra','yadā','tadā','kadā','sadā','punar','tāvat','yāvat',
    'adya','idānīm','nityam','satatam','purā','paścāt','saha','sārdham','vinā',
    'ṛte','prati','anu','antar','bahis','ūrdhvam','adhas','upari','agre',
}

def _norm(s: str) -> str:
    return s.lower().replace('√', '').strip(" '’‍").strip()

def _is_finite_verb(a: dict) -> bool:
    lex   = a.get('lexical_category', '').lower()
    morph = a.get('morphological_and_syntactical_analysis', '').lower()
    return 'verb' in lex and any(p in morph for p in ('first', 'second', 'third'))

def _is_core_noun(a: dict) -> bool:
    lex   = a.get('lexical_category', '').lower()
    morph = a.get('morphological_and_syntactical_analysis', '').lower()
    return 'noun' in lex and any(c in morph for c in ('nominative', 'accusative', 'vocative'))

def choose_pos(word_iast: str, analyses: list):
    surf  = _norm(word_iast)
    bases = {_norm(a.get('base_word_', '')) for a in analyses}
    forms = bases | {surf}
    if forms & PRON_WORDS:
        return 'PRON'
    if forms & IND_WORDS:
        return 'IND'
    if surf.endswith('tvā') or surf.endswith('tum'):
        return 'PART'
    if any(_is_finite_verb(a) for a in analyses) and not any(_is_core_noun(a) for a in analyses):
        return 'VERB'
    return simplify_pos(analyses[0].get('lexical_category', ''))


examples      = []
skipped_files = 0

for path in sorted(json_paths):
    try:
        chapter_data = json.loads(open(path, encoding='utf-8').read())
    except Exception:
        skipped_files += 1
        continue

    for verse in chapter_data:
        verse_dev = clean_verse(verse.get('devanagari', '') or '')
        for item in verse.get('verse_Syn', []):
            if not isinstance(item, dict) or 'analyses' not in item:
                continue
            word_iast = item.get('word', '').strip()
            if not word_iast or not item['analyses']:
                continue
            pos_label = choose_pos(word_iast, item['analyses'])
            if pos_label is None:
                continue
            examples.append({
                'word'     : iast_to_devanagari(word_iast),
                'word_iast': word_iast,
                'verse'    : verse_dev,
                'label'    : pos_label,
            })

print(f"Total training examples : {len(examples):,}")
print(f"Files skipped (errors)  : {skipped_files}")
print()
print("Label distribution after disambiguation:")
for lbl, c in Counter(e['label'] for e in examples).most_common():
    print(f"  {lbl:<6} {c:>7,}  {c/len(examples)*100:5.1f}%")

## Step 4 — Label mappings and train/val split

In [ ]:
label_counts = Counter(e['label'] for e in examples)
print("Label distribution:")
print(f"{'Label':<8} {'Count':>8} {'%':>7}")
print('-' * 28)
total = sum(label_counts.values())
for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f"{label:<8} {count:>8,} {count/total*100:>6.1f}%")

labels = sorted(label_counts.keys())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for i, l in enumerate(labels)}
num_labels = len(labels)
print(f"\nNumber of classes: {num_labels}")
print(f"Label order: {labels}")

random.seed(42)
random.shuffle(examples)
split = int(0.9 * len(examples))
train_examples = examples[:split]
val_examples   = examples[split:]
print(f"\nTrain examples : {len(train_examples):,}")
print(f"Val examples   : {len(val_examples):,}")

## Step 5 — Configuration

Flip `MODEL_NAME` to bench a different base model. The same LoRA config and data pipeline are used for all three (the attention submodules `query`/`key`/`value` exist in ALBERT, BERT and RoBERTa alike).

In [ ]:
import torch

# Pick one. The comparison harness (Step 9) loops over all five automatically.
MODEL_NAME = "google/muril-base-cased"
# MODEL_NAME = "ai4bharat/indic-bert"
# MODEL_NAME = "xlm-roberta-base"
# MODEL_NAME = "l3cube-pune/hindi-bert-v2"
# MODEL_NAME = "ai4bharat/IndicBERTv2-MLM-Sam-TLM"

MAX_LEN = 128   # verse context + word fits comfortably
device  = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## Step 6 — Tokenise with verse context

Each example is a sentence pair: sentence A = full verse (Devanagari), sentence B = the target word. `truncation='only_first'` shortens the verse if needed but never cuts the word. `make_hf_dataset` is parameterised by tokenizer so the comparison harness can re-tokenise per model.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

def make_hf_dataset(example_list, tokenizer):
    raw = Dataset.from_list([
        {'verse': e['verse'], 'word': e['word'], 'label': label2id[e['label']]}
        for e in example_list
    ])

    def tokenize_batch(batch):
        return tokenizer(
            batch['verse'],          # sentence A — verse context
            batch['word'],           # sentence B — word to classify
            padding='max_length',
            truncation='only_first', # truncate verse if needed, never the word
            max_length=MAX_LEN,
        )

    return raw.map(tokenize_batch, batched=True, remove_columns=['verse', 'word'])

## Step 7 — Metrics

`macro_f1` (every class weighted equally) is the primary metric — it is not inflated by the dominant NOUN class. `weighted_f1` and accuracy are reported as secondary.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report

def compute_metrics(eval_pred):
    logits, label_ids = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy'   : accuracy_score(label_ids, preds),
        'macro_f1'   : f1_score(label_ids, preds, average='macro'),
        'weighted_f1': f1_score(label_ids, preds, average='weighted'),
    }

## Step 8 — LoRA model + weighted training (single experiment)

`run_experiment(model_name)` builds a LoRA-wrapped classifier for one base model, trains it with the class-weighted loss + label smoothing, evaluates, prints the per-class report, and returns both the trainer and a metrics dict.

- **Class-weighted loss** (`WeightedTrainer`): rare classes (e.g. PRON) get inverse-frequency weights so the model is penalised harder for missing them.
- **LoRA**: `r=16`, `alpha=32`, adapters on `query`/`key`/`value`; the randomly-initialised `classifier` head is trained fully via `modules_to_save`.
- **Higher LR** (`2e-4`) and a couple more epochs than full fine-tuning, as LoRA trains far fewer parameters.

In [ ]:
from torch.nn import CrossEntropyLoss
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

# Inverse-frequency class weights: weight[i] = total / (num_classes * count[i])
counts        = [label_counts[id2label[i]] for i in range(num_labels)]
_total        = sum(counts)
class_weights = torch.tensor(
    [_total / (num_labels * c) for c in counts], dtype=torch.float
).to(device)
print("Class weights (higher = rarer class, penalised harder):")
for lbl, w in zip(labels, class_weights.tolist()):
    print(f"  {lbl:<6} weight={w:.4f}")


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = CrossEntropyLoss(weight=class_weights)
        loss    = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


def run_experiment(model_name: str):
    print(f"\n{'='*60}\n  EXPERIMENT: {model_name}\n{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_ds  = make_hf_dataset(train_examples, tokenizer)
    val_ds    = make_hf_dataset(val_examples,   tokenizer)

    base = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_labels, id2label=id2label, label2id=label2id,
    )
    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=16, lora_alpha=32, lora_dropout=0.1, bias="none",
        target_modules=["query", "key", "value"],
        modules_to_save=["classifier"],   # train the randomly-init head fully
    )
    model = get_peft_model(base, lora_cfg).to(device)
    model.print_trainable_parameters()    # sanity: trainable % should be small

    safe_name = model_name.replace('/', '_')
    training_args = TrainingArguments(
        output_dir                  = f'./lora-{safe_name}',
        num_train_epochs            = 7,
        per_device_train_batch_size = 32,
        per_device_eval_batch_size  = 64,
        learning_rate               = 2e-4,    # LoRA wants a higher LR than full FT
        warmup_ratio                = 0.06,
        weight_decay                = 0.01,
        label_smoothing_factor      = 0.1,
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'macro_f1',
        greater_is_better           = True,
        logging_steps               = 100,
        report_to                   = 'none',
        fp16                        = (device == 'cuda'),
    )

    trainer = WeightedTrainer(
        model           = model,
        args            = training_args,
        train_dataset   = train_ds,
        eval_dataset    = val_ds,
        compute_metrics = compute_metrics,
    )

    print("\nTraining ... watch the 'macro_f1' column (the honest metric).\n")
    trainer.train()

    eval_results = trainer.evaluate()
    raw_preds = trainer.predict(val_ds)
    preds     = np.argmax(raw_preds.predictions, axis=-1)
    print("\nPer-class report:")
    print(classification_report(raw_preds.label_ids, preds, target_names=labels, digits=4))

    metrics = {
        'model'      : model_name,
        'accuracy'   : eval_results['eval_accuracy'],
        'macro_f1'   : eval_results['eval_macro_f1'],
        'weighted_f1': eval_results['eval_weighted_f1'],
    }
    return trainer, tokenizer, metrics

## Step 9 — Comparison harness (all five models)

Trains every base model in turn and prints a summary table. **This runs five full trainings sequentially** — on a T4 it takes a while. If you only want one model, skip this cell and run the single-model cell below instead.

Results are kept in `runs` so the inference/save steps can use any trained model.

In [ ]:
ALL_MODELS = [
    "ai4bharat/indic-bert",
    "google/muril-base-cased",
    "xlm-roberta-base",
    "l3cube-pune/hindi-bert-v2",
    "ai4bharat/IndicBERTv2-MLM-Sam-TLM",
]

runs = {}
summary = []
for name in ALL_MODELS:
    trainer, tokenizer, metrics = run_experiment(name)
    runs[name] = {'trainer': trainer, 'tokenizer': tokenizer, 'metrics': metrics}
    summary.append(metrics)
    torch.cuda.empty_cache()

print("\n" + "=" * 64)
print("LoRA BENCH SUMMARY (sorted by macro_f1)")
print("=" * 64)
print(f"{'Model':<28} {'Acc':>8} {'MacroF1':>9} {'WtdF1':>8}")
print('-' * 64)
for m in sorted(summary, key=lambda x: -x['macro_f1']):
    print(f"{m['model']:<28} {m['accuracy']:>8.4f} {m['macro_f1']:>9.4f} {m['weighted_f1']:>8.4f}")

### (Alternative) Single-model run

Use this instead of the harness above to train just the model set in `MODEL_NAME` (Step 5).

In [ ]:
# trainer, tokenizer, metrics = run_experiment(MODEL_NAME)
# runs = {MODEL_NAME: {'trainer': trainer, 'tokenizer': tokenizer, 'metrics': metrics}}
# print(metrics)

## Step 10 — Inference

`predict_pos` accepts an optional verse context (matches training conditions and improves accuracy). Set `BEST_MODEL` to whichever run you want to use.

In [ ]:
# Pick the model to run inference with (defaults to the best macro_f1 run).
BEST_MODEL = max(runs, key=lambda k: runs[k]['metrics']['macro_f1'])
print(f"Using: {BEST_MODEL}")
model     = runs[BEST_MODEL]['trainer'].model
tokenizer = runs[BEST_MODEL]['tokenizer']
model.eval()

def predict_pos(word_iast: str, verse_iast: str = '') -> tuple:
    word_dev = iast_to_devanagari(word_iast)
    if verse_iast.strip():
        verse_dev = iast_to_devanagari(verse_iast)
        inputs = tokenizer(verse_dev, word_dev, return_tensors='pt',
                           max_length=MAX_LEN, padding='max_length', truncation='only_first')
    else:
        inputs = tokenizer(word_dev, return_tensors='pt',
                           max_length=MAX_LEN, padding='max_length', truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs      = torch.softmax(logits, dim=-1)[0]
    pred_id    = probs.argmax().item()
    return id2label[pred_id], probs[pred_id].item(), word_dev


test_cases = [
    ('dharma',   '',                                                          'NOUN'),
    ('gacchati', 'tatra tatrāñjasā āyuṣman bhavatā yad viniścitam',          'VERB'),
    ('uvāca',    'śrīśuka uvāca varīyān eṣa te praśnaḥ kṛto lokahitaṃ nṛpa','VERB'),
    ('eṣa',      'śrīśuka uvāca varīyān eṣa te praśnaḥ kṛto lokahitaṃ nṛpa','PRON'),
    ('ca',       'anvayāt itarataḥ ca artheṣu abhijñaḥ svarāṭ',             'IND'),
    ('kṛṣṇa',     '',                                                          'NOUN'),
]

print("─" * 72)
print(f"{'Word':<14} {'Predicted':>10}  {'Conf':>7}  {'Expected':>10}  {'?':>4}")
print("─" * 72)
for word_iast, verse_iast, expected in test_cases:
    label, conf, dev = predict_pos(word_iast, verse_iast)
    correct = "✓" if label == expected else "✗"
    ctx     = "(with context)" if verse_iast else "(no context)"
    print(f"{word_iast:<14} {label:>10}  {conf:>6.1%}  {expected:>10}  {correct}  {ctx}")

## Step 11 — Save the LoRA adapter

`model.save_pretrained` writes **only the adapter** (a few MB) plus `adapter_config.json`.

**To reload later:**
```python
from transformers import AutoModelForSequenceClassification
from peft import PeftModel
base = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=5)
model = PeftModel.from_pretrained(base, SAVE_DIR)
```
Or call `model.merge_and_unload()` first to bake the adapter into the base weights and save a standalone model.

In [ ]:
import shutil

SAVE_DIR = 'sanskrit-pos-lora-adapter'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

with open(f'{SAVE_DIR}/label_map.json', 'w') as f:
    json.dump({'id2label': id2label, 'label2id': label2id, 'base_model': BEST_MODEL}, f, indent=2)

shutil.make_archive('sanskrit_pos_lora', 'zip', SAVE_DIR)
files.download('sanskrit_pos_lora.zip')
print(f"Saved LoRA adapter for {BEST_MODEL} and started download.")